In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table("workspace.bronze.erp_cust_az12")
display(df)

In [0]:
df.select([F.sum(F.col(c).isNull().cast("int")) for c in df.columns]).show()

In [0]:
df = df.withColumn("CID", F.substring(F.col("CID"), F.length(F.col("CID"))-4, F.length(F.col("CID"))))
display(df)

In [0]:
df = df.withColumn(
    "BDATE",
    F.when(F.col("BDATE") < F.lit('1900-01-01'), None)
    .when(F.col("BDATE") > F.current_date(), None)
    .otherwise(F.col("BDATE"))   
)

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(F.col("GEN")).isin("MALE", "M"), "Male")
    .when(F.upper(F.col("GEN")).isin("FEMALE", "F"), "Female") 
    .otherwise("Unknown")
)

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "BDATE": "birth_date",
    "GEN": "gender"
    }
df = df.withColumnsRenamed(RENAME_MAP)
display(df)

In [0]:
df.groupby("gender").count().show()

In [0]:
display(df)

In [0]:
df.select("birth_date").agg(F.min("birth_date"), F.max("birth_date")).show()

In [0]:
df.write.format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customers
LIMIT 10